# 🎙️ Shiro Voice Changer Real-Time (RVC MMVC Server)
Notebook resmi untuk menjalankan **Real-Time Voice Changer (RVC)** Shiro di Google Colab dengan akselerasi GPU (T4 / A100).

### 🌟 Dua Colab Resmi Shiro LLMA Project:
1. 🧠 [**Shiro_LLMA_Colab.ipynb**](https://colab.research.google.com/github/Renn-devBI/ShiroLLM-AI-CHAT/blob/main/Shiro_LLMA_Colab.ipynb) — Otak Utama AI Percakapan, Multimodal Vision, & API Gateway.
2. 🎙️ [**Shiro_VoiceChanger_Colab.ipynb**](https://colab.research.google.com/github/Renn-devBI/ShiroLLM-AI-CHAT/blob/main/Shiro_VoiceChanger_Colab.ipynb) — Voice Changer Real-Time RVC (Notebook ini).

> [!TIP]
> Anda dapat menjalankan kedua notebook ini secara bersamaan di 2 tab Colab terpisah untuk mendapatkan pengalaman VTuber AI penuh: Shiro AI merespons teks/suara dan suara Anda diubah menjadi suara karakter secara real-time!

## Langkah 1: Cek & Verifikasi GPU
Jalankan cell ini untuk memastikan Colab terhubung ke GPU (T4 atau A100).

In [ ]:
import subprocess
res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if res.returncode == 0:
    print("=" * 60)
    print("GPU AKTIF: Siap untuk Voice Changer Real-Time!")
    print("=" * 60)
    print(res.stdout.split('\n')[0])
    for line in res.stdout.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line or 'NVIDIA' in line:
            print(line)
    print("=" * 60)
else:
    print("=" * 60)
    print("PERINGATAN: GPU BELUM DIAKTIFKAN!")
    print("Voice Changer membutuhkan GPU untuk inferensi real-time rendah latency.")
    print("Buka menu Runtime -> Change runtime type -> Pilih T4 GPU -> Simpan.")
    print("=" * 60)

## Langkah 2: Unduh Voice Changer & Pasang Dependensi
Tahap ini akan mengunduh versi terbaru w-okada Voice Changer dan menginstall dependensi audio (FAISS GPU CUDA 12, PyWorld, Fairseq, ONNXRuntime GPU).

Proses instalasi membutuhkan waktu sekitar **2-3 menit**.

In [ ]:
# ==============================================================================
# @title **[1]** Clone Repository & Install Dependencies
# @markdown Step pertama akan mengunduh Voice Changer dan memasang dependensi bebas error.
# ==============================================================================
import os
import sys
import time
import shutil
import subprocess
import threading
import codecs

# 1. Konfigurasi Lingkungan (Colab / Kaggle / Local)
os.chdir('/content' if os.path.exists('/content') else '.')
notebook_env = 1 if os.path.exists('/content') else (2 if os.path.exists('/kaggle/working') else 3)
env_name = 'Google Colab' if notebook_env == 1 else ('Kaggle' if notebook_env == 2 else 'Local/Other')
print(f"> Lingkungan Terdeteksi: {env_name}")

# @markdown ---
# @title **[Opsional]** Hubungkan Google Drive
# @markdown Aktifkan opsi ini untuk menyimpan model suara RVC secara permanen di Google Drive agar tidak hilang saat runtime terputus.
Use_Drive = False # @param {type:"boolean"}

if notebook_env == 1 and Use_Drive:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    drive_model_dir = "/content/drive/MyDrive/Shiro_VoiceChanger/model_dir"
    os.makedirs(drive_model_dir, exist_ok=True)
    print(f"> [Drive] Penyimpanan Model Permanen: {drive_model_dir}")

# 2. Pasang Paket Sistem & Audio Libraries C/C++ (PortAudio, libsndfile)
print("> [1/6] Memasang paket sistem (portaudio, libportaudio2, libsndfile, ffmpeg)...\n")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq build-essential python3-dev libportaudio2 portaudio19-dev libsndfile1 psmisc ffmpeg > /dev/null 2>&1
!npm install -g @hrzn/cli > /dev/null 2>&1 || true

# 3. Kunci pip < 24.1 & NumPy 1.x untuk Kompatibilitas Fairseq & PyWorld
print("> [2/6] Mengunci pip < 24.1, NumPy < 2.0.0 & dependensi build...")
!python -m pip install "pip<24.1" --quiet
!pip install "setuptools<70.0.0" wheel Cython colorama --quiet
!pip install "numpy<2.0.0,>=1.24.0" --quiet

# 4. Clone Repository Voice Changer (Aman Dijalankan Berulang)
print("> [3/6] Mengunduh repository Voice Changer...")
externalgit = codecs.decode('uggcf://tvguho.pbz/j-bxnqn/ibvpr-punatre.tvg', 'rot_13')
pathloc = codecs.decode('ibvpr-punatre', 'rot_13')

os.chdir('/content' if notebook_env == 1 else '.')
if not os.path.exists(pathloc):
    !git clone --depth 1 $externalgit
else:
    print(f"  Folder {pathloc} sudah ada.")

os.chdir(f"{pathloc}/server")

# Modifikasi tag versi UI
version_file = '../client/demo/dist/assets/gui_settings/version.txt'
if os.path.exists(version_file):
    !sed -i "s/-.-.-.-/Shiro.Voice.Mod/" {version_file}

# 5. Pasang FAISS GPU (CUDA 12) & ONNXRuntime GPU Resmi
print("> [4/6] Memasang FAISS GPU (CUDA 12) & ONNXRuntime GPU...")
!pip install faiss-gpu-cu12 --quiet || pip install faiss-cpu --quiet
!pip install onnxruntime-gpu --quiet

# 6. Pasang PyWorld & Fairseq (dengan omegaconf & hydra-core yang kompatibel)
print("> [5/6] Memasang PyWorld & Fairseq...")
!pip install "omegaconf==2.0.6" "hydra-core==1.0.7" --quiet
!pip install pyworld --no-build-isolation --quiet
!pip install fairseq --no-build-isolation --quiet || pip install git+https://github.com/facebookresearch/fairseq.git --no-build-isolation --quiet

# 7. Bersihkan requirements.txt dan Pasang Pustaka Audio Lengkap (sounddevice, librosa, dll)
print("> [6/6] Memasang dependensi server Voice Changer & modul audio...")
!sed -i -E '/(torch|torchaudio|numpy|onnxruntime|scipy)/d' requirements.txt
!sed -i '/^mat$/d' requirements.txt
!sed -i -E 's/==.*//g' requirements.txt
!pip install -r requirements.txt --quiet
!pip install "scipy<1.14" sounddevice soundfile librosa torchcrepe torchfcpe noisereduce pedalboard gdown portpicker pyngrok --quiet
!pip install "numpy<2.0.0,>=1.24.0" --quiet

# Sambungkan direktori model ke Google Drive jika diaktifkan
if notebook_env == 1 and Use_Drive:
    drive_dir = "/content/drive/MyDrive/Shiro_VoiceChanger/model_dir"
    if os.path.exists('model_dir') and not os.path.islink('model_dir'):
        shutil.rmtree('model_dir', ignore_errors=True)
    if not os.path.exists('model_dir'):
        os.symlink(drive_dir, 'model_dir')
else:
    os.makedirs('model_dir', exist_ok=True)

from IPython.display import clear_output
clear_output()
Run_Cell = 1
print("\033[92m=================================================================\033[0m")
print("\033[92m> [SUKSES] Seluruh dependensi Voice Changer berhasil terpasang!\033[0m")
print("\033[92m=================================================================\033[0m")
print("\033[96m> Lanjutkan ke Langkah 3 untuk memasukkan model suara (opsional),\033[0m")
print("\033[96m  atau langsung ke Langkah 4 untuk menjalankan server Voice Changer.\033[0m")

## Langkah 3: Unggah / Impor Model Suara RVC (Opsional)
Gunakan cell ini jika Anda ingin mengimpor model suara karakter RVC (file `.zip` atau `.pth` dari HuggingFace atau Google Drive).

> [!NOTE]
> Anda juga bisa mengunggah model langsung lewat antarmuka Web UI Voice Changer setelah server berjalan pada Langkah 4.

In [ ]:
# ==============================================================================
# @title **[2] [Opsional]** Upload / Impor Model Suara RVC ke Slot Tertentu
# ==============================================================================
import os
import sys
import json
import zipfile
import shutil
import gdown

# Pastikan berada di direktori server
if os.path.exists('/content/voice-changer/server'):
    os.chdir('/content/voice-changer/server')

# Pastikan sounddevice & audio dependencies terpasang sebelum memanggil paramset
try:
    import sounddevice
except ImportError:
    !pip install sounddevice soundfile librosa --quiet

model_slot = 1 # @param {type:"number"}
model_slot = str(model_slot)

# @markdown **[Opsional]** Link gambar icon model (URL gambar png/jpg):
icon_link = "https://i.scdn.co/image/ab6761610000e5ebbab6de14398281b2d37bc0b5" # @param {type:"string"}

# @markdown **Link Model RVC:** Mendukung HuggingFace (direct / resolve link) atau Google Drive:
model_link = "https://huggingface.co/megaaziib/my-rvc-models-collection/resolve/main/zetaTest.zip?download=true" # @param {type:"string"}

# @markdown **Pengaturan Pitch & Indeks Suara:**
Tune = 12 # @param {type:"slider", min:-50, max:50, step:1}
Index = 0 # @param {type:"slider", min:0, max:1, step:0.1}

slot_dir = f"model_dir/{model_slot}"
os.makedirs(slot_dir, exist_ok=True)

# 1. Bersihkan slot lama jika ada berkas usang
for item in os.listdir(slot_dir):
    item_p = os.path.join(slot_dir, item)
    if os.path.isfile(item_p):
        os.remove(item_p)
    elif os.path.isdir(item_p):
        shutil.rmtree(item_p, ignore_errors=True)

# 2. Unduh Berkas Model
print(f"> Mengunduh model RVC untuk Slot {model_slot}...")
model_zip_path = "model.zip"
if os.path.exists(model_zip_path):
    os.remove(model_zip_path)

if model_link.startswith("https://www.weights.gg") or model_link.startswith("https://weights.gg"):
    print("\033[93m[Info] Link weights.gg tidak lagi didukung langsung. Gunakan link HuggingFace atau Google Drive.\033[0m")
elif model_link.startswith("https://drive.google.com"):
    print("  Mengunduh dari Google Drive...")
    gdown.download(model_link, model_zip_path, quiet=False, fuzzy=True)
elif "huggingface.co" in model_link:
    print("  Mengunduh dari HuggingFace...")
    !curl -L -s "{model_link}" -o "{model_zip_path}"
else:
    print("  Mengunduh dari tautan langsung...")
    !curl -L -s "{model_link}" -o "{model_zip_path}"

# 3. Ekstrak Berkas Model (.pth, .onnx, .index) Secara Otomatis & Mandiri
if os.path.exists(model_zip_path) and os.path.getsize(model_zip_path) > 1000:
    try:
        with zipfile.ZipFile(model_zip_path, 'r') as zf:
            extracted_count = 0
            for member in zf.infolist():
                fname = os.path.basename(member.filename)
                if fname.endswith(('.pth', '.onnx', '.index')):
                    dest_file = os.path.join(slot_dir, fname)
                    with zf.open(member) as source, open(dest_file, "wb") as target:
                        shutil.copyfileobj(source, target)
                    extracted_count += 1
            print(f"  Berhasil mengekstrak {extracted_count} berkas model (.pth/.index).")
    except zipfile.BadZipFile:
        shutil.move(model_zip_path, os.path.join(slot_dir, "model.pth"))
    finally:
        if os.path.exists(model_zip_path):
            os.remove(model_zip_path)
else:
    print("\033[93m[Warning] Berkas zip tidak ditemukan atau berukuran 0 bytes. Periksa link unduhan.\033[0m")

# 4. Unduh Icon Gambar
icon_file_name = ""
clean_icon = icon_link.strip().strip('"').strip("'")
if clean_icon:
    icon_file_name = "icon.png"
    icon_target = os.path.join(slot_dir, icon_file_name)
    try:
        !curl -L -s "{clean_icon}" -o "{icon_target}"
    except Exception:
        icon_file_name = ""

# 5. Tulis Skrip paramset.py Lokal (Mandiri Tanpa Pastebin yang Rawan Blokir)
paramset_script = '''import argparse
import os
import json
from voice_changer.RVC.RVCModelSlotGenerator import RVCModelSlotGenerator
from voice_changer.VoiceChangerParamsManager import VoiceChangerParamsManager
from voice_changer.utils.LoadModelParams import LoadModelParamFile, LoadModelParams
from voice_changer.utils.VoiceChangerParams import VoiceChangerParams

parser = argparse.ArgumentParser()
parser.add_argument('--notebook_env', type=str, default='1')
parser.add_argument('--model_slot', type=str, required=True)
parser.add_argument('--iconFile', type=str, default='')
parser.add_argument('--Tune', type=str, default='0')
parser.add_argument('--Index', type=str, default='0')
args = parser.parse_args()

slot_dir = os.path.join("model_dir", args.model_slot)
pth_files = [f for f in os.listdir(slot_dir) if f.endswith(".pth") or f.endswith(".onnx")]
index_files = [f for f in os.listdir(slot_dir) if f.endswith(".index")]

model_file = pth_files[0] if pth_files else ""
model_name = os.path.splitext(model_file)[0] if model_file else f"Model_Slot_{args.model_slot}"
index_file = index_files[0] if index_files else ""

sample_rate = 40000
try:
    vcparams = VoiceChangerParamsManager.get_instance()
    vcparams.setParams(VoiceChangerParams(model_dir=slot_dir))
    file_info = LoadModelParamFile(name=model_file, kind="rvcModel", dir="")
    load_param = LoadModelParams(voiceChangerType="RVC", files=[file_info], slot=args.model_slot, isSampleMode=False, sampleId="", params={})
    slot_info = RVCModelSlotGenerator.loadModel(load_param)
    sample_rate = getattr(slot_info, "samplingRate", 40000)
except Exception as e:
    pass

params_content = {
    "slotIndex": int(args.model_slot) if args.model_slot.isdigit() else 1,
    "voiceChangerType": "RVC",
    "name": model_name,
    "description": "Model Suara Shiro RVC",
    "credit": "",
    "termsOfUseUrl": "",
    "iconFile": args.iconFile,
    "speakers": {"0": "target"},
    "modelFile": model_file,
    "indexFile": index_file,
    "defaultTune": int(float(args.Tune)),
    "defaultIndexRatio": float(args.Index),
    "defaultProtect": 0.5,
    "isONNX": model_file.endswith(".onnx"),
    "modelType": "pyTorchRVCv2",
    "samplingRate": sample_rate,
    "f0": True,
    "embChannels": 768,
    "embOutputLayer": 12,
    "useFinalProj": False,
    "deprecated": False,
    "embedder": "hubert_base",
    "sampleId": "",
    "version": "v2"
}

with open(os.path.join(slot_dir, "params.json"), "w", encoding="utf-8") as f:
    json.dump(params_content, f, indent=2)

print(f"> Parameter slot {args.model_slot} ({model_name}) berhasil dibuat!")
'''

with open('paramset.py', 'w', encoding='utf-8') as f:
    f.write(paramset_script)

!python paramset.py --model_slot "{model_slot}" --iconFile "{icon_file_name}" --Tune "{Tune}" --Index "{Index}" --notebook_env "{notebook_env}"

print(f"\033[92m> [SUKSES] Model suara berhasil disiapkan pada Slot {model_slot}!\033[0m")

## Langkah 4: Jalankan Server Voice Changer
Pilih opsi tunneling yang diinginkan:
- **CLOUDFLARE (Rekomendasi ⭐)**: 100% Gratis, tanpa perlu daftar akun/token, dan tidak memakan kuota session Ngrok.
- **NGROK**: Jika Anda memiliki token dari [dashboard.ngrok.com](https://dashboard.ngrok.com).
- **HRZN**: Jika Anda memiliki token dari [hrzn.run](https://hrzn.run).

Saat pertama kali dijalankan, server akan mengunduh model embedding dasar (~1-2 menit). Tunggu hingga link URL publik muncul!

In [ ]:
# ==============================================================================
# @title **[3]** Jalankan Server Menggunakan **CLOUDFLARE**, **NGROK**, atau **HRZN**
# ==============================================================================
import os
import sys
import time
import socket
import shutil
import threading
import subprocess
import codecs
import portpicker
from IPython.display import clear_output, display, Audio

# 1. Pastikan berada di direktori server voice-changer
if os.path.exists('/content/voice-changer/server'):
    os.chdir('/content/voice-changer/server')

# 2. Verifikasi & Pasang Otomatis Modul Audio Jika Belum Lengkap (sounddevice, librosa, dll)
try:
    import sounddevice
    import librosa
    import soundfile
except ImportError:
    print("> [Auto-Fix] Memasang modul audio yang dibutuhkan (sounddevice, librosa, soundfile)...")
    !apt-get update -qq > /dev/null 2>&1
    !apt-get install -y -qq libportaudio2 portaudio19-dev libsndfile1 > /dev/null 2>&1
    !pip install sounddevice soundfile librosa torchcrepe torchfcpe noisereduce pedalboard --quiet

# ====================== Konfigurasi Tunnel ======================
TUNNEL = "CLOUDFLARE" # @param ["CLOUDFLARE", "NGROK", "HRZN"]
Token = "" # @param {type:"string"}
Region = "ap - Asia/Pacific (Singapore)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)", "eu - Europe (Frankfurt)", "in - India (Mumbai)", "jp - Japan (Tokyo)", "sa - South America (Sao Paulo)", "us - United States (Ohio)"]
ClearConsole = True # @param {type:"boolean"}
Play_Notification = False # @param {type:"boolean"}

PORT = portpicker.pick_unused_port()
public_url = ""

# 3. Inisialisasi Tunnel Publik
if TUNNEL == "CLOUDFLARE":
    print("> Mengaktifkan Cloudflare Tunnel (100% Gratis, Tanpa Token)...\n")
    if not os.path.exists('/content/cloudflared'):
        !curl -s -L -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /content/cloudflared
    
    cf_log = '/content/cf_tunnel.log'
    if os.path.exists(cf_log):
        os.remove(cf_log)
    
    # Jalankan cloudflared di background
    os.system(f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} > {cf_log} 2>&1 &")
    
    # Polling URL publik
    import re
    start_t = time.time()
    while time.time() - start_t < 25:
        if os.path.exists(cf_log):
            with open(cf_log, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                matches = re.findall(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', content)
                if matches:
                    public_url = matches[0]
                    break
        time.sleep(1)

elif TUNNEL == "NGROK":
    from pyngrok import conf, ngrok
    if not Token.strip():
        Token = input("Masukkan Ngrok Authtoken (dari dashboard.ngrok.com): ").strip()
    MyConfig = conf.PyngrokConfig()
    MyConfig.auth_token = Token.strip()
    MyConfig.region = Region[0:2]
    conf.set_default(MyConfig)
    try:
        ngrok.kill()
    except Exception:
        pass
    ngrokConnection = ngrok.connect(PORT)
    public_url = ngrokConnection.public_url

elif TUNNEL == "HRZN":
    !rm -rf url.txt
    !hrzn login $Token
    os.system(f"hrzn tunnel http://localhost:{PORT} >> url.txt 2>&1 &")
    time.sleep(5)
    with open('url.txt', 'r') as file:
        import re
        m = re.findall(r'https://[a-zA-Z0-9.-]+\.hrzn\.run', file.read())
        public_url = m[0] if m else ""

# 4. Penamaan Server Script (Aman Dijalankan Berulang)
mainpy = codecs.decode('ZZIPFreireFVB.cl', 'rot_13') # MMVCServerSIO.py
if os.path.exists(mainpy):
    if os.path.exists("HVoice.py"):
        os.remove("HVoice.py")
    shutil.move(mainpy, "HVoice.py")
    !sed -i "s/MMVCServerSIO/HVoice/g" HVoice.py

os.makedirs("pretrain", exist_ok=True)
os.makedirs("model_dir", exist_ok=True)

# 5. Background Thread untuk Notifikasi Server Siap
def wait_for_server():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        sock.close()
        if result == 0:
            break
    if ClearConsole:
        clear_output()
    print("=" * 64)
    print("  \033[92m\033[1m🎙️ SERVER VOICE CHANGER BERHASIL BERJALAN!\033[0m")
    print("=" * 64)
    print("  Akses Web UI Voice Changer:")
    print(f"  👉 \033[94m\033[1m{public_url}\033[0m")
    print("=" * 64)
    print("  1. Buka link di atas di browser Anda.")
    print("  2. Pilih microphone & speaker di Web UI.")
    print("  3. Tekan tombol START untuk mulai mengubah suara secara real-time!")
    print("=" * 64)
    if Play_Notification:
        display(Audio(url='https://raw.githubusercontent.com/hinabl/rmvpe-ai-kaggle/main/custom/audios/notif.mp3', autoplay=True))

threading.Thread(target=wait_for_server, daemon=True).start()

# 6. Jalankan Voice Changer Server CLI
allowed_origins_arg = f'--allowed-origins "{public_url}" --allowed-origins "*"' if public_url else '--allowed-origins "*"'

!python3 HVoice.py \
    -p {PORT} \
    --https False \
    --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
    --content_vec_500_onnx pretrain/content_vec_500.onnx \
    --content_vec_500_onnx_on false \
    --hubert_base pretrain/hubert_base.pt \
    --hubert_base_jp pretrain/rinna_hubert_base_jp.pt \
    --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
    --nsf_hifigan pretrain/nsf_hifigan/model \
    --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
    --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
    --rmvpe pretrain/rmvpe.pt \
    --model_dir model_dir \
    --samples samples.json \
    {allowed_origins_arg}

# 7. Cleanup Saat Server Berhenti
if TUNNEL == "NGROK":
    try:
        ngrok.kill()
    except Exception:
        pass
elif TUNNEL == "CLOUDFLARE":
    !pkill -f cloudflared || true
elif TUNNEL == "HRZN":
    !fuser -k ${PORT}/tcp 2>/dev/null || true
print("\n--------- SERVER VOICE CHANGER SELESAI ---------")